# Juggle counter with feet + ball fusion

This is the advanced version. It runs two models: one finds the ball, one finds the feet (pose).
A juggle is a foot touch: the ball reaches a low point close to a foot. A break is when the ball leaves the feet,
for example a failed trick where the ball drops to the ground. We count the juggles and split them into streaks.

It is about two times slower than the simple version, because it runs two models per frame.
Run each cell with Shift+Enter.

## 1. Imports

Here we import the libraries we need.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from scipy.signal import find_peaks

print('ok')

## 2. Load the two models

Here we load the ball detector (yolov8s) and the pose model (yolov8s-pose). The pose model gives 17 body points.
The ankles are points 15 (left) and 16 (right).

In [ ]:
det = YOLO('yolov8s.pt')
pose = YOLO('yolov8s-pose.pt')
SPORTS_BALL = 32
L_ANKLE, R_ANKLE = 15, 16
print('models ready')

## 3. Track the ball and the feet across the video

Here we run both models on every frame. For the ball we store the box and its center and size.
For the feet we store the two ankle positions. This is the slow part.

In [ ]:
VIDEO = 'jungle_video.mp4'

cap = cv2.VideoCapture(VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

ball_cx, ball_cy, ball_diam = [], [], []
ball_box = []                 # (x1,y1,x2,y2) or None, reused for the video
foot_lx, foot_rx, foot_y = [], [], []
feet_pts = []                 # list of ankle points per frame, reused for the video

while True:
    ok, frame = cap.read()
    if not ok:
        break
    # ball
    dr = det(frame, classes=[SPORTS_BALL], conf=0.25, verbose=False)[0]
    if len(dr.boxes) > 0:
        j = int(dr.boxes.conf.argmax())
        x1, y1, x2, y2 = dr.boxes.xyxy[j].tolist()
        ball_box.append((x1, y1, x2, y2))
        ball_cx.append((x1 + x2) / 2); ball_cy.append((y1 + y2) / 2); ball_diam.append(y2 - y1)
    else:
        ball_box.append(None)
        ball_cx.append(np.nan); ball_cy.append(np.nan); ball_diam.append(np.nan)
    # feet
    pr = pose(frame, conf=0.3, verbose=False)[0]
    lx = ly = rx = ry = np.nan
    pts = []
    if pr.keypoints is not None and len(pr.keypoints) > 0:
        kp = pr.keypoints.data[0]
        a, b, ca = kp[L_ANKLE].tolist()
        c, dd, cb = kp[R_ANKLE].tolist()
        if ca > 0.3: lx, ly = a, b; pts.append((int(a), int(b)))
        if cb > 0.3: rx, ry = c, dd; pts.append((int(c), int(dd)))
    foot_lx.append(lx); foot_rx.append(rx)
    foot_y.append(np.nanmax([ly, ry]) if not (np.isnan(ly) and np.isnan(ry)) else np.nan)
    feet_pts.append(pts)

cap.release()
ball_cx = np.array(ball_cx); ball_cy = np.array(ball_cy); ball_diam = np.array(ball_diam)
foot_lx = np.array(foot_lx); foot_rx = np.array(foot_rx)
print('frames processed:', len(ball_cx))

## 4. Build the height signal

Here we turn the ball center into a height (H minus y, so up is a big number), fill the gaps and smooth a little.

In [ ]:
def fill(a):
    a = a.copy(); nan = np.isnan(a)
    if nan.any():
        a[nan] = np.interp(np.flatnonzero(nan), np.flatnonzero(~nan), a[~nan])
    return a

cx = fill(ball_cx); cy = fill(ball_cy); diam = fill(ball_diam)
flx = fill(foot_lx); frx = fill(foot_rx)
s = np.convolve(H - cy, np.ones(3) / 3, mode='same')
times = np.arange(len(s)) / fps

## 5. Find the touches (low points)

Here we find the low points of the ball trajectory. Each low point is a candidate touch.
We use a robust amplitude (the 10th to 90th percentile range) instead of the full max minus min.
The full range is inflated by one very high trick, which would hide the fast low juggles.

In [ ]:
robust_amplitude = np.percentile(s, 90) - np.percentile(s, 10)
min_gap = int(fps * 0.15)
valleys, _ = find_peaks(-s, distance=min_gap, prominence=0.10 * robust_amplitude)
print('candidate touches:', len(valleys))

## 6. Fusion: is the ball at a foot?

Here is the fusion step. At each low point we measure the horizontal distance from the ball to the nearest ankle,
in units of ball diameters (so it does not depend on the camera distance).
If the ball is close to a foot it is a real juggle. If it is far, the ball left the feet, which is a break.

In [ ]:
NEAR = 1.2   # ball within 1.2 ball diameters of a foot counts as a foot touch

juggle_frames, break_frames = [], []
streaks, current = [], 0
for v in valleys:
    dist = min(abs(cx[v] - flx[v]), abs(cx[v] - frx[v])) / diam[v]
    if dist < NEAR:
        current += 1
        juggle_frames.append(v)
    else:
        break_frames.append(v)
        streaks.append(current); current = 0
streaks.append(current)
streaks = [x for x in streaks if x > 0]

for i, st in enumerate(streaks, 1):
    print('streak', i, ':', st, 'juggles')
print('longest streak:', max(streaks))

## 7. Plot the result

Here we plot the height. Green dots are juggles. Red crosses are breaks (the ball left the feet).

In [ ]:
plt.figure(figsize=(13, 4))
plt.plot(times, s, color='steelblue')
plt.plot(times[juggle_frames], s[juggle_frames], 'go', ms=9, label='juggle')
if break_frames:
    plt.plot(times[break_frames], s[break_frames], 'rx', ms=13, mew=3, label='break')
plt.xlabel('time (s)'); plt.ylabel('height'); plt.legend(); plt.grid(True, alpha=0.3)
plt.title('juggles and breaks')
plt.show()

## 8. Annotated video

Here we write a video with the ball box, the feet, and a streak counter that resets on a break.
We reuse the stored positions, so the models do not run again. avc1 is used because mp4v fails on this OpenCV build.

In [ ]:
juggle_set = set(int(f) for f in juggle_frames)
break_set = set(int(f) for f in break_frames)
out_w = 960

cap = cv2.VideoCapture(VIDEO)
src_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
scale = out_w / src_w
out_size = (out_w, int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) * scale))
writer = cv2.VideoWriter('juggle_fusion.mp4', cv2.VideoWriter_fourcc(*'avc1'), fps, out_size)

count = 0
flash = 0
i = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    if i in juggle_set:
        count += 1
    if i in break_set:
        count = 0; flash = int(fps)   # show DROP for about 1 second
    frame = cv2.resize(frame, out_size)
    if ball_box[i] is not None:
        x1, y1, x2, y2 = [int(v * scale) for v in ball_box[i]]
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
    for (fx, fy) in feet_pts[i]:
        cv2.circle(frame, (int(fx * scale), int(fy * scale)), 8, (0, 255, 255), -1)
    cv2.putText(frame, 'Streak: ' + str(count), (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0, 255, 0), 3, cv2.LINE_AA)
    if flash > 0:
        cv2.putText(frame, 'DROP', (20, 110), cv2.FONT_HERSHEY_SIMPLEX, 1.6, (0, 0, 255), 4, cv2.LINE_AA)
        flash -= 1
    writer.write(frame)
    i += 1

cap.release()
writer.release()
print('saved juggle_fusion.mp4')